# Version of the Kaggle ECG Classification Problem with 
* Multi-Head Classifier
* EMA weights for validation model
* No segmentation
* reduceLRonPlateau() scheduler

## Read in data and test-train-splitting

In [ ]:
import os

import pandas as pd
import numpy as np
import random

import torch 
from torch import nn 
from torch.utils.data import Dataset, DataLoader

import torchvision.models as models
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split

import cv2 
from PIL import Image

from tqdm import tqdm

In [ ]:
data_path = os.path.abspath("../../../data")
labels_df = pd.read_csv(os.path.join(data_path, "train_final.csv"), index_col=0, dtype=int)

images_list_path = os.path.abspath("../../../broken_images")
with open(os.path.join(images_list_path, "broken_images_list.txt"), "r") as file:
    broken_train_images = [line.strip() for line in file]
with open(os.path.join(images_list_path, "broken_test_images_list.txt"), "r") as file:
    broken_test_images = [line.strip() for line in file]
with open(os.path.join(images_list_path ,"valid_images_list.txt"), "r") as file:
    valid_train_images = [line.strip() for line in file]
with open(os.path.join(images_list_path, "valid_test_images_list.txt"), "r") as file:
    valid_test_images = [line.strip() for line in file]

all_ids = set([int(obj.split(".")[-2][-6:]) for obj in valid_train_images])

In [ ]:
import glob 
local_files = glob.glob(os.path.join(data_path, "train", "train_*"))
indices = set([int(item.split(".")[0][-6:]) for item in local_files])

# usable local ids 
local_ids = indices.intersection(all_ids)

# downsample ids to be more tractable numbers
ids = [id_val for ind, id_val in enumerate(local_ids) if ind%10 == 0]

print(len(local_ids))
print(len(ids))

In [ ]:

# rewrite "valid" images to have useful paths 
valid_images_local = [os.path.join(data_path, "train", path.split("/")[-1]) for path in valid_train_images if int(path.split(".")[-2][-6:]) in ids]

labels_df = labels_df.loc[labels_df.index.isin(ids)]

# split train set, using stratification 
X_train, X_test, y_train, y_test = train_test_split(valid_images_local, 
                                                    labels_df, 
                                                    test_size = 0.1,
                                                    random_state = 42, 
                                                    shuffle = True,
                                                    stratify = labels_df[["AF", "HYP"]])

In [ ]:
mean_vals = labels_df.to_numpy().mean(axis=0) 
pos_weight = (1 - mean_vals)/mean_vals
pos_weight_tensor = torch.from_numpy(pos_weight).clip(min=None, max=10.0).to(dtype=torch.float32)

## Dataset and DataLoader

In [ ]:
def smart_pad_and_resize_ecg(image, target_size=(224, 224)):
    """
    Smarter padding that detects background color automatically
    Generated from Claude
    """
    if isinstance(image, Image.Image):
        image = np.array(image)
    
    # Auto-detect background color (assume corners are background)
    corners = [
        image[0, 0], image[0, -1], 
        image[-1, 0], image[-1, -1]
    ]
    
    # Use most common corner color as background
    if len(image.shape) == 3:
        bg_color = np.median(corners, axis=0).astype(image.dtype)
    else:
        bg_color = np.median(corners).astype(image.dtype)
    
    h, w = image.shape[:2]
    max_dim = max(h, w)
    
    # Create padded canvas
    if len(image.shape) == 3:
        padded = np.full((max_dim, max_dim, image.shape[2]), bg_color, dtype=image.dtype)
    else:
        padded = np.full((max_dim, max_dim), bg_color, dtype=image.dtype)
    
    # Center the image
    y_offset = (max_dim - h) // 2
    x_offset = (max_dim - w) // 2
    padded[y_offset:y_offset + h, x_offset:x_offset + w] = image

    # sharpen image
    kernel = np.array([[-0.5, -0.5, -0.5],
                       [-0.5,  5.0, -0.5],
                       [-0.5, -0.5, -0.5]])
    
    sharpened = cv2.filter2D(padded, -1, kernel)
    # Blend original and sharpened (subtle effect)
    enhanced = cv2.addWeighted(padded, 0.7, sharpened, 0.3, 0)
    
    # Resize with appropriate interpolation
    resized = cv2.resize(padded, target_size, interpolation=cv2.INTER_AREA)
    
    return resized

In [ ]:
class ECG_Dataset(Dataset):
    def __init__(self, 
                 image_paths: list[str], 
                 label_df: pd.DataFrame, 
                 transforms=None, 
                 new_width=512, 
                 new_height=512, 
                 dtype=torch.float32) -> None:
        """
        Custom Pytorch Dataset Class for the ECG problem
        Utilises a custom resizing function which pads first
        Also needs to do relatively complex label extraction because labels are saved as dataframes
        """
        
        self.image_paths = image_paths
        self.labels_df = label_df
        self.transforms = transforms
        self.new_width = new_width 
        self.new_height = new_height
        self.dtype = dtype

    def __len__(self) -> int: 
        return len(self.image_paths)

    def __getitem__(self, idx) -> tuple[torch.Tensor, torch.Tensor]:
        image_path = self.image_paths[idx]
        image = cv2.imread(image_path)
        
        if image is None:
            raise ValueError(f"Failed to load image: {image_path}")

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        image_resize = smart_pad_and_resize_ecg(
            image_rgb, target_size=(self.new_width, self.new_height)
        )

        image_tens = torch.from_numpy(image_resize).to(self.dtype)

        if self.transforms:
            image_tens = self.transforms(image_tens)

        filename = os.path.basename(image_path)
        index = int(filename.rsplit('_', 1)[-1].split('.')[0])
        row = self.labels_df[self.labels_df.index == index].values
        label = torch.tensor(row, dtype=self.dtype)
                
        return image_tens, label

In [ ]:
class FastPaperFoldEffect(object):
    """Simplified paper fold effect"""
    def __init__(self, max_intensity=0.3):
        self.max_intensity = max_intensity
    
    def __call__(self, img_tensor):        
        # Create single vertical or horizontal fold
        h, w = img_tensor.shape[1:]
        # is_vertical = random.random() > 0.5
        fold_pos = random.uniform(0.2, 0.8)

        fold_pos_px = int(w * fold_pos)
        img_tensor[:, :, fold_pos_px-2:fold_pos_px+2] *= random.uniform(0.7, 0.9)
        
        return img_tensor

test_transforms = transforms.Compose([
    transforms.Lambda(lambda x: x.permute(2, 0, 1)),  # HWC → CHW
    transforms.ConvertImageDtype(torch.float32),      # Ensure float32
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet normalization
                         std=[0.229, 0.224, 0.225]),
])

train_transforms = transforms.Compose([
    # HWC → CHW
    transforms.Lambda(lambda x: x.permute(2, 0, 1)),
    transforms.ConvertImageDtype(torch.float32),

    # Geometric augmentations
    #transforms.RandomRotation(degrees=5),                  # small angle shift
    #transforms.RandomResizedCrop(512, scale=(0.9, 1.0)),   # random crop + resize
    #transforms.RandomApply([transforms.GaussianBlur(3)], p=0.1),  # occasional blur
    #transforms.RandomApply([FastPaperFoldEffect(max_intensity=0.1)], p=0.2), # Randomly adds a fold to the paper 


    # Normalize for EfficientNet
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


In [ ]:
train_dataset = ECG_Dataset(
                    X_train, 
                    labels_df, 
                    transforms=train_transforms, 
                    new_width=512, 
                    new_height=512
                    )

test_dataset = ECG_Dataset(
                    X_test, 
                    labels_df, 
                    transforms=test_transforms, 
                    new_width=512, 
                    new_height=512
                    )

In [ ]:
g = torch.Generator()
g.manual_seed(42)

batch_size = 4

train_dataloader = DataLoader(
                    train_dataset, 
                    num_workers = 0,
                    shuffle = True, 
                    batch_size = batch_size,
                    pin_memory=True,
                    generator = g
                    )

test_dataloader = DataLoader(
                    test_dataset, 
                    num_workers = 0,
                    batch_size = batch_size,
                    pin_memory=True,
                    )

## Defining Core Model Structure

In [ ]:
class Head(nn.Module):
    def __init__(self, in_features, hidden_layer, dropout_rate=0.3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden_layer),
            nn.BatchNorm1d(hidden_layer),  
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer, hidden_layer // 2), 
            nn.BatchNorm1d(hidden_layer // 2),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer // 2, 1)
        )
    
    def forward(self, x):
        return self.layers(x)

class MultiHeadEfficientNet(nn.Module):
    def __init__(self, num_conditions=5, hidden_dim=512, dropout_rate=0.3):
        super().__init__()
        
        backbone = models.efficientnet_v2_s(weights="DEFAULT")
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        
        self.backbone = backbone
        
        self.shared_feature_processor = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )
        
        self.heads = nn.ModuleList([
            Head(hidden_dim, hidden_dim // 2, dropout_rate) 
            for _ in range(num_conditions)
        ])
        
    def forward(self, x):
        backbone_feats = self.backbone(x)
        processed_feats = self.shared_feature_processor(backbone_feats)
        outputs = [head(processed_feats) for head in self.heads]
        return torch.cat(outputs, dim=1)  # [batch, num_conditions]

## Define EMA of Model Weights (for validation)

In [ ]:
# import copy
# from collections import OrderedDict

In [ ]:
# class ModelEMA:
#     def __init__(self, model, decay=0.9999, warmup_steps=2000, update_freq=1):
#         """
#         Enhanced EMA implementation with warmup and configurable update frequency
        
#         Args:
#             model: The model to track
#             decay: EMA decay rate (higher = more smoothing)
#             warmup_steps: Number of steps to linearly increase decay from 0 to target
#             update_freq: Update EMA every N steps (for computational efficiency)
#         """
#         # Make a copy of the model for EMA
#         self.ema_model = copy.deepcopy(model)
#         self.ema_model.eval()
        
#         # EMA parameters should not require gradients
#         for p in self.ema_model.parameters():
#             p.requires_grad_(False)
        
#         self.decay = decay
#         self.warmup_steps = warmup_steps
#         self.update_freq = update_freq
#         self.step_count = 0
        
#         # Store original model reference for easy access
#         self.model = model

#     def get_current_decay(self):
#         """Calculate current decay rate with warmup"""
#         if self.step_count < self.warmup_steps:
#             # Linear warmup from 0 to target decay
#             return self.decay * (self.step_count / self.warmup_steps)
#         return self.decay

#     def update(self, model=None):
#         """Update EMA weights"""
#         if model is None:
#             model = self.model
            
#         self.step_count += 1
        
#         # Only update every update_freq steps for efficiency
#         if self.step_count % self.update_freq != 0:
#             return
            
#         current_decay = self.get_current_decay()
        
#         with torch.no_grad():
#             model_state_dict = model.state_dict()
#             ema_state_dict = self.ema_model.state_dict()
            
#             for key in ema_state_dict.keys():
#                 if key in model_state_dict:
#                     ema_param = ema_state_dict[key]
#                     model_param = model_state_dict[key].detach()
                    
#                     # Apply EMA update
#                     ema_param.copy_(
#                         current_decay * ema_param + (1.0 - current_decay) * model_param
#                     )

#     def copy_to_model(self, model=None):
#         """Copy EMA weights to the original model"""
#         if model is None:
#             model = self.model
            
#         model.load_state_dict(self.ema_model.state_dict())

#     def state_dict(self):
#         """Return EMA model state dict for saving"""
#         return {
#             'ema_model': self.ema_model.state_dict(),
#             'decay': self.decay,
#             'step_count': self.step_count,
#             'warmup_steps': self.warmup_steps,
#             'update_freq': self.update_freq
#         }
    
#     def load_state_dict(self, state_dict):
#         """Load EMA state dict"""
#         self.ema_model.load_state_dict(state_dict['ema_model'])
#         self.decay = state_dict['decay']
#         self.step_count = state_dict['step_count']
#         self.warmup_steps = state_dict['warmup_steps']
#         self.update_freq = state_dict['update_freq']

## Metrics for Validation

In [ ]:
def calculate_metrics(outputs, labels, alpha=0.8):
    """
    Calculate multiple metrics for multi-label classification
    Our model outputs a quantity for each separate value, so we evaluate them separately
    Vectorised calculation of these quantities, keeping them all on the GPU
    """

    assert isinstance(outputs, torch.Tensor) and isinstance(labels, torch.Tensor), \
    f"Outputs and labels must both be torch.Tensor objects, got {type(outputs)} and {type(labels)}"
    # Convert outputs to predictions (0 or 1)
    predictions = (torch.sigmoid(outputs) > 0.5).int()
    labels = labels.squeeze().int()

    tp = (predictions & labels).sum(axis=0)  # true positives per class
    fp = (predictions & (~labels)).sum(axis=0)  # false positives
    fn = ((~predictions) & labels).sum(axis=0)  # false negatives
    
    # F1 per class
    f1s = 2*tp / (2*tp + fp + fn + 1e-8)  # add epsilon to avoid div by zero
    
    # Accuracy per class
    accs = (predictions == labels).float().mean(axis=0)
    
    # Hamming loss per class
    hammings = (predictions != labels).float().mean(axis=0)

    macro_f1 = f1s.mean()
    min_f1 = f1s.min()
    # smoothed average of macro and minimum F1 score for reporting
    combined_f1 = alpha * macro_f1 + (1 - alpha) * min_f1

    return {
        "accuracy": accs,
        "f1": f1s,
        "hamming": hammings,
        "macro_f1": macro_f1,
        "min_f1": min_f1,
        "combined_f1": combined_f1
    }

## Training Loop 

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import Adam
import warnings

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
if device.type == "cuda" and device.index is None:
    device = torch.device(f"cuda:{torch.cuda.current_device()}")
print(f"Using {device=}")

# hyperparams
learning_rate = 1.0e-2
weight_decay = 5.0e-5
patience = 3
lr_reduce_factor = 0.5
cooldown = 1
lr_reduce_threshold = 1.0e-3
dropout_rate = 0.3 
hidden_layer = 512
max_epochs = 100

model = MultiHeadEfficientNet(
    num_conditions=5, hidden_dim=hidden_layer, dropout_rate=dropout_rate
    ).to(device)

# use default values for beta and eps in Adam
opt = Adam(model.parameters(), weight_decay=weight_decay, lr=learning_rate)

scheduler = ReduceLROnPlateau(opt, 
                            mode='min', 
                            factor=lr_reduce_factor, 
                            patience=patience, 
                            cooldown=cooldown, 
                            threshold=lr_reduce_threshold)

# ema = ModelEMA(
#         model, 
#         decay=0.9995,      # Slightly lower for medical data (more responsive)
#         warmup_steps=1000,  # Warmup for first 1000 steps
#         update_freq=5       # Update EMA every 5 steps for efficiency
#     )

checkpoint_path = "checkpoint.pth"
start_epoch = 0
checkpoint = None
if os.path.exists(checkpoint_path):
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        opt.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        #ema.load_state_dict(checkpoint['ema_state_dict'])
        #ema.ema_model.to(device)
        start_epoch = checkpoint['epoch'] + 1
    except Exception as e:
        print(f"Unable to load checkpoint from path {checkpoint_path}")
else:
    warnings.warn(f"Checkpoint path {checkpoint_path} not found, not restarting using checkpoint")

In [ ]:
# model_device = next(model.parameters()).device
# #ema_device = next(ema.ema_model.parameters()).device

# assert model_device.index == device.index, \
#     f"Model is on {model_device}, expected {device}"
# # assert ema_device.index == device.index, \
# #     f"EMA model is on {ema_device}, expected {device}"

In [ ]:
model_save_path = "checkpoint.pth"

num_classes = 5
num_batches = len(train_dataloader)
assert batch_size != 1, "Batch size cannot be 1"

accumulation_steps = 4

pos_weight_tensor = pos_weight_tensor.to(device)
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor,
)
if checkpoint is not None:
    best_val_loss = checkpoint["val_loss"]
else:
    best_val_loss = float('inf')
patience = 10
patience_counter = 0

max_epochs = 10

train_losses = []
running_training_metrics = []
val_losses = []
running_val_metrics = []

for epoch in range(start_epoch, max_epochs):
    
    for param_group in opt.param_groups:
        usable_lr = param_group['lr']
        print(f"Epoch {epoch+1}: Learning Rate = {usable_lr:.3e}")

    # Training phase
    model.train()
    # running training loss is a tensor to prevent device transfers
    running_train_loss = torch.tensor(0.0, device=device)
    training_metrics = {
        "accuracy": torch.zeros(num_classes, device=device),
        "f1": torch.zeros(num_classes, device=device),
        "hamming": torch.zeros(num_classes, device=device),
        "macro_f1": torch.tensor(0.0, device=device),
        "min_f1": torch.tensor(0.0, device=device),
        "combined_f1": torch.tensor(0.0, device=device)
    }

    opt.zero_grad()
    pbar = tqdm(total=len(train_dataloader), desc=f"Epoch {epoch+1} - Training", unit="batch")
    for i, (inputs, labels) in enumerate(train_dataloader):

        if (i + 1) % 1 == 0 or (i + 1) == len(train_dataloader):
            pbar.n = i + 1
            pbar.refresh()
        
        inputs = inputs.to(device)
        labels = labels.squeeze().to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)/accumulation_steps
        loss.backward()
        
        if (i + 1) % accumulation_steps == 0:
            # clip gradient norm to 1: prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            #ema.update()
            opt.zero_grad()

        # detach loss to prevent gradients continuing into running_train_loss
        running_train_loss = running_train_loss + (loss.detach() * accumulation_steps)
        batch_metrics = calculate_metrics(outputs.detach(), labels.detach())
        training_metrics["accuracy"] = training_metrics["accuracy"] + batch_metrics["accuracy"]
        training_metrics["f1"] = training_metrics["f1"] + batch_metrics["f1"]
        training_metrics["hamming"] = training_metrics["hamming"] + batch_metrics["hamming"]
        training_metrics["macro_f1"] = training_metrics["macro_f1"] + batch_metrics["macro_f1"]
        training_metrics["min_f1"] = training_metrics["min_f1"] + batch_metrics["min_f1"]
        training_metrics["combined_f1"] = training_metrics["combined_f1"] + batch_metrics["combined_f1"]
    
    if (i + 1) % accumulation_steps != 0:
        # update in the edge case where num_batches is not divisible by accumulation_steps
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        #ema.update()
        opt.zero_grad()

    pbar.close()
    
    num_batches = len(train_dataloader)
    # calculate average training metrics
    avg_train_loss = running_train_loss.item() / num_batches
    training_metrics = {k: v.cpu().numpy() / num_batches for k, v in training_metrics.items()}
    print(f"\nEpoch [{epoch+1}/{max_epochs}] Training Metrics:")
    print(f"Loss: {avg_train_loss:.4f}")
    print(f"Macro F1: {training_metrics['macro_f1']:.4f}")
    print(f"Combined F1: {training_metrics['combined_f1']:.4f}")
    print("Per-class F1 Scores:", ' '.join(f"{x:.4f}" for x in training_metrics['f1']))
    train_losses.append(avg_train_loss)
    running_training_metrics.append(training_metrics)

    # VALIDATION TIME 
    #ema.ema_model.eval()
    model.eval()
    running_val_loss = torch.tensor(0.0, device=device)
    val_metrics = {
        "accuracy": torch.zeros(num_classes, device=device),
        "f1": torch.zeros(num_classes, device=device),
        "hamming": torch.zeros(num_classes, device=device),
        "macro_f1": torch.tensor(0.0, device=device),
        "min_f1": torch.tensor(0.0, device=device),
        "combined_f1": torch.tensor(0.0, device=device)
    }
    with torch.inference_mode():
        pbar = tqdm(total=len(test_dataloader), desc=f"Epoch {epoch+1} - Testing", unit="batch")
        for i, (inputs, labels) in enumerate(test_dataloader):
            if (i + 1) % 1 == 0 or (i + 1) == len(test_dataloader): 
                # updates the progress bar every 100 batches 
                pbar.n = i+1
                pbar.refresh()
                        
            inputs = inputs.to(device)
            labels = labels.squeeze().to(device)
            # no gradient tracking
            # use EMA model to do validation
            #outputs = ema.ema_model(inputs)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
    
            running_val_loss = running_val_loss + loss 
            batch_metrics = calculate_metrics(outputs, labels)
            val_metrics["accuracy"] = val_metrics["accuracy"] + batch_metrics["accuracy"]
            val_metrics["f1"] = val_metrics["f1"] + batch_metrics["f1"]
            val_metrics["hamming"] = val_metrics["hamming"] + batch_metrics["hamming"]
            val_metrics["macro_f1"] = val_metrics["macro_f1"] + batch_metrics["macro_f1"]
            val_metrics["min_f1"] = val_metrics["min_f1"] + batch_metrics["min_f1"]
            val_metrics["combined_f1"] = val_metrics["combined_f1"] + batch_metrics["combined_f1"]
    pbar.close()
    num_val_batches = len(test_dataloader)
    # calculate average validation metrics
    avg_val_loss = running_val_loss.item() / num_val_batches
    val_metrics = {k: v.cpu().numpy() / num_val_batches for k, v in val_metrics.items()}
    print(f"\nEpoch [{epoch+1}/{max_epochs}] Validation Metrics:")
    print(f"Loss: {avg_val_loss:.4f}")
    print(f"Macro F1: {val_metrics['macro_f1']:.4f}")
    print(f"Combined F1: {val_metrics['combined_f1']:.4f}")
    print("Per-class F1 Scores:", ' '.join(f"{x:.4f}" for x in val_metrics['f1']))
    val_losses.append(avg_val_loss)
    running_val_metrics.append(val_metrics)

    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        
        # Save both regular model and EMA model
        torch.save({
            'model_state_dict': model.state_dict(),
            #'ema_state_dict': ema.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'epoch': epoch + 1,
            'learning_rate': usable_lr,
            'val_loss': val_losses,
            'train_loss': train_losses,
            'train_metrics': running_training_metrics,
            'val_metrics': running_val_metrics
        }, model_save_path)
        
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch}')
            break